# Streaming detection: results analysis

Streaming object-detection runs on the `cityday_curated` manifest (13
ordered domain blocks, 2 000-frame city-day bootstrap).  The analysis
compares five filter families against the no-filter upper bound:

- `static` -- bootstrap-only Mahalanobis threshold (no refresh).
- `window` -- Mahalanobis with sliding window of the last M accepted frames.
- `reservoir` -- Mahalanobis with fixed-size reservoir (Vitter's Algorithm R).
- `uncertainty` -- detection-head uncertainty, adaptive or static.
- `random` -- compute-matched null baseline at multiple accept rates.

The filter should react to domain shifts, adapt to new domains, and thin
accepts over time while matching no-filter mAP at lower compute.

Sections:

1. **Setup** -- run discovery, bootstrap composition.
2. **Summary tables** -- per-(variant, seed) metrics and cross-seed aggregate.
3. **mAP along the stream** -- per-checkpoint mAP with domain blocks marked.
4. **Multi-seed mAP comparison** -- mean / min / max mAP across seeds.
5. **Iso-compute mAP** -- mAP at a matched optimizer-step budget.
6. **Per-domain mAP** -- mean mAP inside each manifest block.
7. **Pedestrian AP** -- class-level domain-shift indicator.
8. **Forgetting analysis** -- early vs. late per-class AP.
9. **Accept-rate dynamics** -- rolling accept rate along the stream.
10. **Accept rate between refreshes** -- reaction + within-block decay diagnostic.
11. **Per-block accept rate** -- accept rate inside each manifest block.
12. **Per-domain acceptance** -- accept rate by scene bucket.
13. **Cumulative accepts** -- cumulative accepted frames along the stream.
14. **Acceptance-rate breakdown** -- metadata composition and conditional rates.
15. **Score separation** -- accepted vs rejected filter scores.
16. **Category-level acceptance** -- per-class accept fraction.
17. **Inter-accept gaps** -- temporal thinning vs memoryless random.
18. **Object-count scatter** -- accepted frames cover denser scenes.
19. **Scoring-model refresh timeline** -- threshold evolution over refreshes.

**Data**: `checkpoints.csv`, `streaming_metrics.csv`, `filter_stats.csv`, `decisions.csv`, `refreshes.csv`, and the manifest referenced by each run's `config.yaml`.

See `02_federated_analysis.ipynb` for federated (multi-client) experiments.

## 1 Setup

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.axes import Axes
import numpy as np
import pandas as pd

%matplotlib inline

sys.path.insert(0, str(Path.cwd() / "notebooks"))
if not (Path.cwd() / "pyproject.toml").exists():
    sys.path.insert(0, str(Path.cwd().parent / "notebooks"))

import analysis_helpers as ah

ah.setup_notebook_style()

PROJECT_ROOT = ah.find_project_root()
OUTPUTS = PROJECT_ROOT / "outputs"

SEED: int | None = None
SUBSAMPLE_N = 12_000
RNG = 42

# Variant -> color for per-variant plots.  Filter-family plots use
# analysis_helpers.FILTER_FAMILY_COLORS instead, which keeps a stable
# color per filter family across variants.
PALETTE: dict[str, str] = {
    # Reference
    "no_filter_cityday_curated": "#2ca02c",
    # Mahalanobis family
    "static_p15_cityday_curated": "#1f77b4",
    "adaptive_window_p15_cityday_curated": "#ff7f0e",
    "adaptive_window_p15_m2000_cityday_curated": "#f4a261",
    "adaptive_reservoir_p15_cityday_curated": "#d62728",
    "adaptive_reservoir_p20_cityday_curated": "#8c0000",
    # Reservoir size and refresh-cadence sweep (refresh every 10 flushes)
    "adaptive_reservoir_p15_fast_cityday_curated": "#e377c2",
    "adaptive_reservoir_p15_r1000_fast_cityday_curated": "#9467bd",
    "adaptive_reservoir_p15_r4000_fast_cityday_curated": "#8c564b",
    # Uncertainty family
    "uncertainty_p15_cityday_curated": "#17becf",
    "adaptive_uncertainty_p15_cityday_curated": "#0d7480",
    "adaptive_uncertainty_p20_cityday_curated": "#184f55",
    # Compute-matched random baselines
    "random_p17_cityday_curated": "#a0a0a0",
    "random_p21_cityday_curated": "#909090",
    "random_p29_cityday_curated": "#7f7f7f",
    "random_p73_cityday_curated": "#555555",
}

print("Project:", PROJECT_ROOT)

In [ ]:
runs_df = ah.discover_runs(OUTPUTS)


def pick(pipeline: str, variant: str, seed: int | None = SEED) -> Path | None:
    return ah.pick_latest_run(runs_df, pipeline, variant, seed=seed)


def _runs_non_null(raw: dict[str, Path | None]) -> dict[str, Path]:
    out: dict[str, Path] = {}
    for k, v in raw.items():
        if v is not None:
            out[k] = v
    return out


# Variants in the primary RUN dict.  Non-existent variants are silently
# skipped, so the list can include runs that have not finished yet --
# only directories present under outputs/streaming/ show up downstream.
_VARIANTS: list[str] = [
    # Reference
    "no_filter_cityday_curated",
    # Mahalanobis family (static and adaptive)
    "static_p15_cityday_curated",
    "adaptive_window_p15_cityday_curated",
    "adaptive_window_p15_m2000_cityday_curated",
    "adaptive_reservoir_p15_cityday_curated",
    "adaptive_reservoir_p20_cityday_curated",
    # Reservoir size and refresh-cadence sweep (refresh every 10 flushes)
    "adaptive_reservoir_p15_fast_cityday_curated",
    "adaptive_reservoir_p15_r1000_fast_cityday_curated",
    "adaptive_reservoir_p15_r4000_fast_cityday_curated",
    # Uncertainty family (static and adaptive)
    "uncertainty_p15_cityday_curated",
    "adaptive_uncertainty_p15_cityday_curated",
    "adaptive_uncertainty_p20_cityday_curated",
    # Compute-matched random baselines (accept rates matched to filter runs)
    "random_p17_cityday_curated",
    "random_p21_cityday_curated",
    "random_p29_cityday_curated",
    "random_p73_cityday_curated",
]
RUN: dict[str, Path] = _runs_non_null({v: pick("streaming", v) for v in _VARIANTS})

MANIFESTS: dict[str, dict] = {}
for k, p in sorted(RUN.items()):
    cfg = ah.load_run_config(p)
    ordering = "?"
    man = ah.load_manifest(PROJECT_ROOT, str(cfg.get("manifest_path", "")))
    if man:
        ordering = ah.ordering_summary(man)
        MANIFESTS[k] = man
    print(f"{k:40s}  seed={cfg.get('seed')}  ordering={ordering}")
    print(f"{'':40s}  {p}")

MANIFEST = next(iter(MANIFESTS.values()), None)

In [ ]:
# --- Seeds available per variant ---
# When a variant has multiple seeds, downstream plots can aggregate across
# seeds using ah.pick_runs_by_seed / ah.aggregate_across_seeds.  The RUN
# dict above holds the *latest* run per variant (at a single seed) and is
# what most cells in this notebook still use.
SEEDS_PER_VARIANT: dict[str, list[int]] = {
    v: ah.discover_seeds(runs_df, "streaming", v) for v in RUN
}
for v, seeds in SEEDS_PER_VARIANT.items():
    tag = "multi-seed" if len(seeds) > 1 else "single"
    print(f"  {v:40s}  seeds={seeds}  [{tag}]")

MULTI_SEED = {v: seeds for v, seeds in SEEDS_PER_VARIANT.items() if len(seeds) >= 2}
print(f"\n{len(MULTI_SEED)} variants have >=2 seeds.")

In [ ]:
# Load enriched decisions (manifest merge only -- no ZOD for speed).
# Set SKIP_ZOD=False and ensure STREAM_ACTIVE_FL_ZOD_ROOT is set to
# add scene_bucket columns for road-type analysis.
SKIP_ZOD = True
_zod = None if SKIP_ZOD else os.environ.get("STREAM_ACTIVE_FL_ZOD_ROOT")

ENRICHED: dict[str, pd.DataFrame] = {}
for key, rdir in RUN.items():
    df = ah.load_enriched_streaming_decisions(rdir, PROJECT_ROOT, zod_root=_zod)
    if not df.empty:
        ENRICHED[key] = df
        print(f"{key:30s}  {len(df):>6d} rows")

### Bootstrap composition

In [ ]:
from collections import Counter

_BOOTSTRAP_NAMES: dict[str, str] = {
    "cityday": "City-Day (city + day)",
    "citymix": "City-Mixed (city, proportional ToD)",
}

def _bootstrap_tag(variant: str) -> str:
    """Infer the bootstrap type from the variant name."""
    if "cityday" in variant:
        return "cityday"
    if "citymix" in variant:
        return "citymix"
    return "unknown"

for k, man in MANIFESTS.items():
    cfg = ah.load_run_config(RUN[k])
    boot_n = ah.get_bootstrap_size(man, cfg)
    boot = ah.bootstrap_train_frames(man, boot_n)
    if not boot:
        continue

    boot_tag = _bootstrap_tag(k)
    boot_label = _BOOTSTRAP_NAMES.get(boot_tag, boot_tag)

    rt_dist = Counter(f.get("road_type", "?") for f in boot)
    tod_dist = Counter(f.get("time_of_day", "?") for f in boot)
    ped_count = sum(1 for f in boot if "Pedestrian" in (f.get("categories_present") or []))

    print(f"\n{k}  (bootstrap={boot_label}, n={len(boot)})")
    print(f"  road_type:    {dict(rt_dist)}")
    print(f"  time_of_day:  {dict(tod_dist)}")
    print(f"  Pedestrian:   {ped_count}/{len(boot)} ({ped_count/len(boot)*100:.1f}%)")

## 2 Summary tables

In [ ]:
# Per-(variant, seed) summary: filter family, manifest, accept rate,
# best / last / iso-compute mAP, refresh count, wall-clock duration.
# ah.variant_summary_table is also what notebooks/analyze_runs.py consumes,
# so the same metrics are reproducible outside the notebook.
# target_optim_steps is the smallest final optimizer-step count across
# filter runs of each manifest; the no-filter baseline is truncated to that
# budget to give a fair iso-compute mAP.
_filter_variants = [v for v in RUN.keys()
                    if ah.filter_mode(ah.load_run_config(ah.pick_latest_run(runs_df, 'streaming', v)))
                    in {'static', 'window', 'reservoir'}]
_steps_by_manifest: dict[str, int] = {}
for _v in _filter_variants:
    for _rd in ah.pick_runs_by_seed(runs_df, 'streaming', _v).values():
        _cfg = ah.load_run_config(_rd)
        _ck = ah.read_csv(_rd / 'checkpoints.csv')
        _s = ah.compute_step_series(_ck)
        if _s is None or _s.dropna().empty:
            continue
        _m = ah.manifest_family(_cfg)
        _last = int(_s.dropna().iloc[-1])
        _steps_by_manifest[_m] = min(_steps_by_manifest.get(_m, _last), _last)
per_seed = ah.variant_summary_table(
    runs_df, 'streaming', list(RUN.keys()),
    target_optim_steps=_steps_by_manifest or None,
)
if per_seed.empty:
    print("No runs found for the variants in RUN.")
else:
    pretty = per_seed.drop(columns=["run_dir"]).copy()
    for c in ("accept_rate", "best_mAP", "last_mAP", "iso_mAP"):
        if c in pretty:
            pretty[c] = pretty[c].astype(float).round(4)
    print("Per-(variant, seed) summary:")
    print(pretty.to_string(index=False))

    agg = ah.aggregate_summary_across_seeds(per_seed)
    if not agg.empty:
        for c in agg.columns:
            if agg[c].dtype.kind == "f":
                agg[c] = agg[c].round(4)
        print("\nAggregated across seeds:")
        print(agg.to_string(index=False))

## 3 mAP along the stream

In [ ]:
# Per-run block boundary info.  Each variant may reference a manifest
# with its own bootstrap size, so we resolve it per-run rather than
# assuming a single global value.
BLOCK_TRANS_PER_RUN: dict[str, list[tuple[int, str]]] = {}
for k, man in MANIFESTS.items():
    cfg = ah.load_run_config(RUN[k])
    boot_n = ah.get_bootstrap_size(man, cfg)
    bt = ah.block_transitions(man, bootstrap_frames=boot_n)
    BLOCK_TRANS_PER_RUN[k] = bt
    print(f"{k:40s}  {len(bt)} transitions  (bootstrap={boot_n})")

# Domain color / short-name maps come from analysis_helpers so notebooks
# 00, 01, 02 all use the same palette.  Curated-manifest blocks (e.g.
# city_day_clear) have no single canonical color yet, so they fall back to
# the first underscore-segmented token (city_day -> city).
_DOMAIN_COLORS: dict[str, str] = {**ah.DOMAIN_COLORS, **ah.WEATHER_COLORS,
                                  "urban": "#1f77b4", "rural": "#2ca02c",
                                  "other": "#999999"}
_DOMAIN_SHORT: dict[str, str] = {**ah.ROAD_SHORT, **ah.WEATHER_SHORT,
                                 "urban": "Urban", "rural": "Rural", "other": "Other"}


def _fallback_color(label: str) -> str:
    return _DOMAIN_COLORS.get(label, _DOMAIN_COLORS.get(label.split("_", 1)[0], "#cccccc"))


def _fallback_short(label: str) -> str:
    if label in _DOMAIN_SHORT:
        return _DOMAIN_SHORT[label]
    head = label.split("_", 1)[0]
    return _DOMAIN_SHORT.get(head, label[:8])

def add_block_bands(
    ax: Axes,
    variant: str | None = None,
    alpha: float = 0.13,
) -> None:
    """Shade domain-block regions for a specific variant.
    If variant is None, do nothing (useful for cross-manifest comparison plots).
    Bands are clipped to the actual plotted data extent (ax.dataLim).
    """
    if variant is None:
        return
    bt = BLOCK_TRANS_PER_RUN.get(variant, [])
    if not bt:
        return
    data_end = ax.dataLim.x1
    if not np.isfinite(data_end) or data_end <= 0:
        return
    for i, (idx, label) in enumerate(bt):
        end = bt[i + 1][0] if i + 1 < len(bt) else data_end
        if idx >= data_end:
            continue
        end = min(end, data_end)
        color = _fallback_color(label)
        ax.axvspan(idx, end, color=color, alpha=alpha, zorder=0)
        ax.axvline(idx, color=color, alpha=0.35, lw=0.6, ls="--", zorder=0)
        width = end - idx
        if width < 1500:
            continue
        mid = (idx + end) / 2
        ax.text(mid, ax.get_ylim()[1], _fallback_short(label),
                ha="center", va="bottom", fontsize=5.5, style="italic",
                color=color, fontweight="bold")


# Aggregate mAP -- one line per variant, no domain bands (bands would
# overlap across manifests with different block orderings).
variants_main = list(RUN.keys())

fig, ax = plt.subplots(figsize=(10, 4))
for v in variants_main:
    if v not in RUN:
        continue
    ck = ah.read_csv(RUN[v] / "checkpoints.csv")
    if ck is None or ck.empty or "mAP" not in ck.columns:
        continue
    ax.plot(ck["items_processed"], ck["mAP"],
            label=v.replace("_", " "), color=PALETTE.get(v), lw=1.4)
ax.set_xlabel("items processed")
ax.set_ylabel("mAP")
ax.set_title("Streaming mAP per variant")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- Per-class AP ---
ap_cols = None
series_list: list[tuple[str, pd.DataFrame]] = []
for v in variants_main:
    if v not in RUN:
        continue
    ck = ah.read_csv(RUN[v] / "checkpoints.csv")
    if ck is None or ck.empty:
        continue
    cols = ah.per_class_ap_columns(ck)
    if not cols:
        continue
    if ap_cols is None:
        ap_cols = cols
    series_list.append((v, ck))

if series_list and ap_cols is not None:
    plot_cols: list[str] = list(ap_cols)
    n_c = len(plot_cols)
    fig, axes = plt.subplots(1, n_c, figsize=(4.2 * n_c, 3.5), squeeze=False)
    for i, col in enumerate(plot_cols):
        a = axes[0][i]
        for label, ck in series_list:
            if col not in ck.columns:
                continue
            a.plot(ck["items_processed"], ck[col], label=label.replace("_", " "),
                   color=PALETTE.get(label), lw=1.2)
        a.set_title(col.replace("AP_", ""))
        a.set_xlabel("items processed")
        a.set_ylabel("AP")
        a.grid(True, alpha=0.3)
    axes[0][0].legend(fontsize=6, loc="best")
    fig.suptitle("Per-class AP vs stream progress", y=1.02)
    plt.tight_layout()
    plt.show()

## 4 Multi-seed mAP comparison

In [ ]:
multi_seed_variants = [v for v in RUN if len(SEEDS_PER_VARIANT.get(v, [])) >= 2]

if not multi_seed_variants:
    print("No variants with >=2 seeds yet; re-run after the multi-seed campaign.")
else:
    fig, ax = plt.subplots(figsize=(10, 4))
    for v in multi_seed_variants:
        runs_by_seed = ah.pick_runs_by_seed(runs_df, "streaming", v)
        agg = ah.aggregate_across_seeds(runs_by_seed, "checkpoints.csv", "items_processed", "mAP")
        if agg.empty:
            continue
        color = PALETTE.get(v)
        ax.plot(agg["items_processed"], agg["mean"], label=f"{v} (n={len(runs_by_seed)})",
                color=color, lw=1.4)
        ax.fill_between(agg["items_processed"], agg["min"], agg["max"],
                        color=color, alpha=0.20, linewidth=0)
    ax.set_xlabel("items processed")
    ax.set_ylabel("mAP")
    ax.set_title("Streaming mAP -- mean (line) and min/max (band) across seeds")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Per-seed peak mAP summary table
    rows: list[dict] = []
    for v in multi_seed_variants:
        runs_by_seed = ah.pick_runs_by_seed(runs_df, "streaming", v)
        summ = ah.summary_across_seeds(runs_by_seed)
        if summ.empty or "best_mAP" not in summ.columns:
            continue
        mean = summ["best_mAP"].mean()
        std = summ["best_mAP"].std(ddof=0)
        rows.append({
            "variant": v,
            "n_seeds": len(summ),
            "best_mAP_mean": round(mean, 4),
            "best_mAP_std": round(std, 4),
            "best_mAP_seeds": [(int(s), round(m, 4)) for s, m in
                               zip(summ["seed"], summ["best_mAP"])],
        })
    if rows:
        print("\nPeak-mAP across seeds:")
        print(pd.DataFrame(rows).to_string(index=False))

## 5 Iso-compute mAP

In [ ]:
# Iso-compute leaderboard.  For each manifest family, pick the smallest
# final optim-step count across FILTER variants (excluding no_filter and
# random baselines) to use as the iso-compute budget, then report the mAP
# of the last checkpoint with optim_steps <= budget for every run.
iso_rows: list[dict] = []
min_steps_by_manifest: dict[str, int] = {}
for v, rdir in RUN.items():
    ck = ah.read_csv(rdir / "checkpoints.csv")
    if ck is None or ck.empty or "optimizer_steps" not in ck.columns:
        continue
    cfg = ah.load_run_config(rdir)
    man = ah.manifest_family(cfg)
    fam = ah.filter_mode(cfg)
    if fam in {"static", "window", "reservoir", "uncertainty"}:
        prev = min_steps_by_manifest.get(man, 10**12)
        min_steps_by_manifest[man] = min(prev, int(ck["optimizer_steps"].iloc[-1]))

for v, rdir in RUN.items():
    ck = ah.read_csv(rdir / "checkpoints.csv")
    if ck is None or ck.empty:
        continue
    cfg = ah.load_run_config(rdir)
    man = ah.manifest_family(cfg)
    tgt = min_steps_by_manifest.get(man)
    stats = ah.iso_compute_mAP(ck, target_optim_steps=tgt)
    iso_rows.append({
        "variant": v,
        "manifest": man,
        "filter_family": ah.filter_mode(cfg),
        "target_optim_steps": tgt,
        "iso_mAP": stats["iso_mAP"],
        "best_mAP": stats["best_mAP"],
        "last_mAP": stats["last_mAP"],
        "final_optim_steps": int(ck["optimizer_steps"].iloc[-1]),
    })
iso_df = pd.DataFrame(iso_rows).sort_values(["manifest", "iso_mAP"], ascending=[True, False])
for c in ("iso_mAP", "best_mAP", "last_mAP"):
    if c in iso_df.columns:
        iso_df[c] = iso_df[c].astype(float).round(4)
print(iso_df.to_string(index=False))


In [ ]:
# Trajectory: mAP vs optimizer_steps, averaged across seeds per variant.
# Highlights the compute-efficiency gap: at matched steps, some filters beat
# random; at unlimited steps, no_filter dominates.
if runs_df is not None and not runs_df.empty:
    fig, ax = plt.subplots(figsize=(10, 4.5))
    plotted = 0
    for v in sorted(RUN):
        rdir = RUN[v]
        ck = ah.read_csv(rdir / "checkpoints.csv")
        if ck is None or ck.empty or "optimizer_steps" not in ck.columns:
            continue
        # Average across all seeds of this variant at common optim-step bins.
        seed_ck_frames = []
        for _, sd in ah.pick_runs_by_seed(runs_df, "streaming", v).items():
            sk = ah.read_csv(sd / "checkpoints.csv")
            if sk is not None and "optimizer_steps" in sk.columns:
                seed_ck_frames.append(sk[["optimizer_steps", "mAP"]])
        if not seed_ck_frames:
            continue
        merged = pd.concat(seed_ck_frames, ignore_index=True).sort_values("optimizer_steps")
        # Bin, then take mean within bin.
        merged["bin"] = pd.cut(merged["optimizer_steps"], bins=20)
        grp = merged.groupby("bin", observed=True, as_index=False).agg(
            optimizer_steps=("optimizer_steps", "mean"),
            mAP=("mAP", "mean"),
        ).dropna()
        ax.plot(grp["optimizer_steps"], grp["mAP"],
                label=ah.SHORT_NAMES.get(v, v) if hasattr(ah, "SHORT_NAMES") else v,
                color=ah.PALETTE.get(v, None) if hasattr(ah, "PALETTE") else None,
                marker="o", ms=3, lw=1.3, alpha=0.9)
        plotted += 1
    ax.set(xlabel="Optimizer steps", ylabel="Aggregate val mAP",
           title="mAP vs compute -- filters beat random at matched opt steps")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7, loc="lower right", ncol=2)
    plt.tight_layout()
    plt.show()
    print(f"plotted {plotted} variants")


In [ ]:
# best-checkpoint Pareto: best mAP vs optimizer-steps-at-best.  The filter
# variants should sit to the upper-LEFT of no_filter if they reach their peak
# with less compute.
if runs_df is not None and not runs_df.empty:
    pts: list[dict] = []
    for v in sorted(RUN):
        for _, sd in ah.pick_runs_by_seed(runs_df, "streaming", v).items():
            ck = ah.read_csv(sd / "checkpoints.csv")
            if ck is None or ck.empty or "mAP" not in ck.columns:
                continue
            best_idx = ck["mAP"].idxmax()
            pts.append({
                "variant": v,
                "seed": sd.parent.name,
                "best_mAP": float(ck.loc[best_idx, "mAP"]),
                "best_step": float(ck.loc[best_idx, "optimizer_steps"]),
            })
    pdf = pd.DataFrame(pts)
    if pdf.empty:
        print("No best-checkpoint data.")
    else:
        pdf_mean = pdf.groupby("variant", as_index=False).agg(
            best_mAP=("best_mAP", "mean"),
            best_step=("best_step", "mean"),
        )
        fig, ax = plt.subplots(figsize=(8, 5))
        for _, r in pdf_mean.iterrows():
            v = r["variant"]
            ax.scatter(r["best_step"], r["best_mAP"],
                       s=55, color=ah.PALETTE.get(v, None) if hasattr(ah, "PALETTE") else None,
                       edgecolor="k", linewidth=0.4, zorder=3)
            ax.annotate(ah.SHORT_NAMES.get(v, v) if hasattr(ah, "SHORT_NAMES") else v,
                        (r["best_step"], r["best_mAP"]),
                        xytext=(5, 3), textcoords="offset points", fontsize=7)
        ax.set(xlabel="Optimizer steps at best checkpoint",
               ylabel="Best val mAP",
               title="Peak-mAP Pareto: reaching peak at lower compute is better (upper-left)")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        print(pdf_mean.sort_values("best_mAP", ascending=False).round(4).to_string(index=False))


## 6 mAP by stream block

In [ ]:
# Per-domain mAP (using per-variant block transitions to assign checkpoints)
if BLOCK_TRANS_PER_RUN:
    domain_map_rows: list[dict] = []
    all_domains_ordered: list[str] = []
    for v in variants_main:
        if v not in RUN or v not in BLOCK_TRANS_PER_RUN:
            continue
        _bt = BLOCK_TRANS_PER_RUN[v]
        if not _bt:
            continue
        ck = ah.read_csv(RUN[v] / "checkpoints.csv")
        if ck is None or ck.empty or "mAP" not in ck.columns:
            continue
        for _, row in ck.iterrows():
            ip = int(row["items_processed"])  # type: ignore[arg-type]
            domain = _bt[0][1]
            for _j, (idx, lbl) in enumerate(_bt):
                if ip >= idx:
                    domain = lbl
            ped_val = row.get("AP_Pedestrian")
            domain_map_rows.append({
                "variant": v, "domain": domain, "mAP": float(row["mAP"]),
                "AP_Pedestrian": float(ped_val) if ped_val is not None else float("nan"),
            })
        if not all_domains_ordered:
            all_domains_ordered = [lbl for _, lbl in _bt]

    dm_df = pd.DataFrame(domain_map_rows)
    if not dm_df.empty and all_domains_ordered:
        variants_found: list[str] = dm_df["variant"].unique().tolist()
        agg = dm_df.groupby(["variant", "domain"])["mAP"].mean().reset_index()

        fig, ax = plt.subplots(figsize=(max(6, 1.5 * len(all_domains_ordered) * len(variants_found)), 4))
        x = np.arange(len(all_domains_ordered))
        w = 0.8 / max(len(variants_found), 1)
        for i, v in enumerate(variants_found):
            sub = agg[agg["variant"] == v]
            vals = []
            for d in all_domains_ordered:
                sel = sub.loc[sub["domain"] == d, "mAP"]
                vals.append(float(sel.iloc[0]) if not sel.empty else 0.0)
            ax.bar(x + i * w, vals, w, label=v.replace("_", " "),
                   color=PALETTE.get(v, f"C{i}"), alpha=0.8)
        ax.set_xticks(x + w * (len(variants_found) - 1) / 2)
        ax.set_xticklabels(all_domains_ordered, fontsize=8)
        ax.set_ylabel("mean mAP")
        ax.set_title("Per-domain mean mAP (from checkpoints within each block)")
        ax.legend(fontsize=7, loc="best")
        plt.tight_layout()
        plt.show()

        agg_ped = dm_df.groupby(["variant", "domain"])["AP_Pedestrian"].mean().reset_index()
        print("\nPedestrian AP by domain:")
        pivot_ped = agg_ped.pivot(  # type: ignore
            index="domain", columns="variant", values="AP_Pedestrian"
        )
        print(pivot_ped.reindex(all_domains_ordered).to_string(float_format="%.4f"))
else:
    print("No block transitions available.")

## 7 Per-domain val mAP (retrospective)

Per-domain AP on the validation split, broken down by time-of-day,
road condition, and road type.  Uses `per_domain_eval.csv` produced
by `tools/per_domain_eval.py`.  For runs trained with live per-domain
eval wired in, `per_domain_checkpoints.csv` also contains the same
breakdown at every checkpoint.

Note: deltas reported below are at `best` / `final` checkpoints only,
which are NOT iso-compute.  See Section 5 for the compute-matched
picture; the aggregate-mAP ordering at iso-compute is very different
(filters beat random at matched optimizer steps).


In [ ]:
# Load per-domain val eval across variants and seeds.  Produces one
# aggregated row per (run_variant, dimension, bucket) with mean +/- std
# across seeds.  checkpoint="final" uses final_model.pt; pass "best" to
# switch to best-mAP checkpoints.
def _collect_run_dirs(variant: str) -> list[Path]:
    return [p for _, p in ah.pick_runs_by_seed(runs_df, "streaming", variant).items()]


per_dom_variants = [v for v in RUN]
per_dom_run_dirs: list[Path] = []
for v in per_dom_variants:
    per_dom_run_dirs.extend(_collect_run_dirs(v))

PER_DOMAIN_RAW = ah.collect_per_domain_eval(per_dom_run_dirs)
PER_DOMAIN_AGG = ah.aggregate_per_domain_eval(per_dom_run_dirs, checkpoint="final")

if PER_DOMAIN_RAW.empty:
    print("No per_domain_eval.csv files found yet.  Run slurm/per_domain_eval.sbatch to generate.")
else:
    print(f"Loaded {len(PER_DOMAIN_RAW)} per-(run, dim, bucket) rows "
          f"from {PER_DOMAIN_RAW['run_variant'].nunique()} variants.")
    print("Aggregated (variant, dim, bucket):", len(PER_DOMAIN_AGG))

In [ ]:
# Gain vs no_filter baseline (final checkpoints), per (dim, bucket).
# Positive = variant outperforms no_filter on that slice.
BASELINE_VARIANT = "no_filter_cityday_curated"
# Bucket values as recorded in the cityday_curated manifest.  Order controls
# the bar-chart column ordering within each dimension.
HIGHLIGHT_BUCKETS = {
    "time_of_day": ["day", "twilight", "night"],
    "road_condition": ["normal", "wet", "snow"],
    "road_type": ["city", "arterial-urban", "highway", "arterial-rural", "smaller-rural"],
}

if PER_DOMAIN_AGG.empty:
    print("Per-domain aggregation empty; skipping gain plot.")
else:
    gain = ah.per_domain_gain_vs_baseline(PER_DOMAIN_AGG, BASELINE_VARIANT, "mAP_mean")
    if gain.empty or "gain" not in gain.columns:
        print("No gain table available (baseline may be missing).")
    else:
        fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
        for ax, dim in zip(axes, ["time_of_day", "road_condition", "road_type"]):
            sub = gain.loc[gain["dimension"] == dim].copy()
            if sub.empty:
                ax.set_title(f"{dim}: no data")
                continue
            buckets = [b for b in HIGHLIGHT_BUCKETS[dim] if b in sub["bucket"].unique()]
            variants_present = [v for v in per_dom_variants if v in sub["run_variant"].unique()]
            variants_present = [v for v in variants_present if v != BASELINE_VARIANT]
            x = np.arange(len(buckets))
            w = 0.8 / max(len(variants_present), 1)
            for i, v in enumerate(variants_present):
                vals = []
                for b in buckets:
                    sel = sub.loc[(sub["run_variant"] == v) & (sub["bucket"] == b), "gain"]
                    vals.append(float(sel.iloc[0]) if not sel.empty else np.nan)
                ax.bar(x + i * w, vals, w, label=v.replace("_cityday_curated", ""),
                       color=PALETTE.get(v, f"C{i}"), alpha=0.85)
            ax.axhline(0, color="black", lw=0.5)
            ax.set_xticks(x + w * (len(variants_present) - 1) / 2)
            ax.set_xticklabels([ah.short_name(b) for b in buckets], rotation=20, ha="right")
            ax.set_title(dim.replace("_", " "))
            ax.grid(True, axis="y", alpha=0.3)
        axes[0].set_ylabel(f"mAP gain vs {BASELINE_VARIANT.replace('_cityday_curated','')}")
        axes[-1].legend(fontsize=6, loc="best", ncol=1)
        plt.tight_layout()
        plt.show()

In [ ]:
# Per-domain mAP heatmap: rows = variants, cols = (dim, bucket).
if not PER_DOMAIN_AGG.empty:
    dims_order = ["time_of_day", "road_condition", "road_type"]
    pivot_cols: list[tuple[str, str]] = []
    for dim in dims_order:
        for b in HIGHLIGHT_BUCKETS[dim]:
            if ((PER_DOMAIN_AGG["dimension"] == dim) &
                (PER_DOMAIN_AGG["bucket"] == b)).any():
                pivot_cols.append((dim, b))
    matrix = np.full((len(per_dom_variants), len(pivot_cols)), np.nan)
    for i, v in enumerate(per_dom_variants):
        for j, (dim, b) in enumerate(pivot_cols):
            sel = PER_DOMAIN_AGG.loc[
                (PER_DOMAIN_AGG["run_variant"] == v)
                & (PER_DOMAIN_AGG["dimension"] == dim)
                & (PER_DOMAIN_AGG["bucket"] == b),
                "mAP_mean",
            ]
            if not sel.empty:
                matrix[i, j] = float(sel.iloc[0])

    fig, ax = plt.subplots(figsize=(max(8, 0.6 * len(pivot_cols)),
                                    max(3, 0.4 * len(per_dom_variants))))
    im = ax.imshow(matrix, aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(pivot_cols)))
    ax.set_xticklabels([f"{dim[:3]}:{ah.short_name(b)}" for dim, b in pivot_cols],
                       rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(len(per_dom_variants)))
    ax.set_yticklabels([v.replace("_cityday_curated", "") for v in per_dom_variants],
                       fontsize=7)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            if not np.isnan(matrix[i, j]):
                ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center",
                        color="white" if matrix[i, j] < 0.3 else "black", fontsize=6)
    fig.colorbar(im, ax=ax, shrink=0.7, label="mAP")
    ax.set_title("Per-(val) domain mAP (mean across seeds, final checkpoint)")
    plt.tight_layout()
    plt.show()


In [ ]:
# Pareto plot: aggregate val mAP vs night-only mAP (final checkpoint).
# A filter that beats the baseline on night without falling behind on the
# aggregate sits in the top-right quadrant.
if not PER_DOMAIN_AGG.empty:
    agg_map = PER_DOMAIN_AGG.loc[
        PER_DOMAIN_AGG["dimension"] == "aggregate", ["run_variant", "mAP_mean"]
    ].rename(columns={"mAP_mean": "aggregate_mAP"})
    night_map = PER_DOMAIN_AGG.loc[
        (PER_DOMAIN_AGG["dimension"] == "time_of_day")
        & (PER_DOMAIN_AGG["bucket"] == "night"),
        ["run_variant", "mAP_mean"],
    ].rename(columns={"mAP_mean": "night_mAP"})
    pareto = agg_map.merge(night_map, on="run_variant", how="outer")
    if not pareto.empty:
        fig, ax = plt.subplots(figsize=(6.5, 5.5))
        for _, r in pareto.iterrows():
            v = r["run_variant"]
            ax.scatter(r["aggregate_mAP"], r["night_mAP"],
                       color=PALETTE.get(v, "#888"), s=60,
                       edgecolors="black", linewidths=0.4)
            ax.annotate(v.replace("_cityday_curated", ""),
                        (r["aggregate_mAP"], r["night_mAP"]),
                        fontsize=6, xytext=(3, 3), textcoords="offset points")
        ax.set_xlabel("aggregate val mAP")
        ax.set_ylabel("night val mAP")
        ax.set_title("Pareto: night-domain gain vs aggregate mAP")
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()


In [ ]:
# Balanced and worst-bucket mAP per dimension, per variant.
if not PER_DOMAIN_AGG.empty:
    bal = ah.balanced_map_table(PER_DOMAIN_AGG, metric="mAP_mean")
    for dim in ["time_of_day", "road_condition", "road_type"]:
        sub = bal.loc[bal["dimension"] == dim].copy()
        if sub.empty:
            continue
        sub = sub.sort_values("balanced_mAP", ascending=False).reset_index(drop=True)
        sub["variant"] = sub["run_variant"].str.replace("_cityday_curated", "", regex=False)
        display_cols = ["variant", "balanced_mAP", "worst_mAP", "n_buckets"]
        print(f"\n== {dim} ==")
        print(sub[display_cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))


## 8 Pedestrian AP

In [ ]:
# Pedestrian AP vs stream progress (headline class)
_ped_col = "AP_Pedestrian"
fig, ax = plt.subplots(figsize=(9, 4))

for v in variants_main:
    if v not in RUN:
        continue
    ck = ah.read_csv(RUN[v] / "checkpoints.csv")
    if ck is None or ck.empty or _ped_col not in ck.columns:
        continue
    ax.plot(ck["items_processed"], ck[_ped_col],
            label=v.replace("_", " "), color=PALETTE.get(v), lw=1.4)

ax.set_xlabel("items processed")
ax.set_ylabel("AP (Pedestrian)")
ax.set_title("Pedestrian AP vs stream progress")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9 Forgetting analysis

In [ ]:
forget_rows = []
for v in variants_main:
    if v not in RUN:
        continue
    ck = ah.read_csv(RUN[v] / "checkpoints.csv")
    if ck is None or ck.empty:
        continue
    cols = ah.per_class_ap_columns(ck)
    if not cols:
        continue
    ft = ah.forgetting_table(ck, cols, n_bins=4)
    if ft.empty:
        continue
    for cls_col, row in ft.iterrows():
        forget_rows.append({
            "variant": v,
            "class": str(cls_col).replace("AP_", ""),
            "early_AP": row["early"],
            "late_AP": row["late"],
            "delta": row["delta"],
        })

if forget_rows:
    fdf = pd.DataFrame(forget_rows)
    print(fdf.to_string(index=False, float_format="%.4f"))

    fig, ax = plt.subplots(figsize=(8, 3.5))
    classes = fdf["class"].unique()
    variants = fdf["variant"].unique()
    x = np.arange(len(classes))
    w = 0.8 / len(variants)
    for i, v in enumerate(variants):
        sub = fdf[fdf["variant"] == v]
        deltas = []
        for c in classes:
            sel = sub.loc[sub["class"] == c, "delta"]
            deltas.append(float(sel.iloc[0]) if not sel.empty else 0.0)
        ax.bar(x + i * w, deltas, w, label=v.replace("_", " "),
               color=PALETTE.get(v, "#999"), alpha=0.8)
    ax.set_xticks(x + w * (len(variants) - 1) / 2)
    ax.set_xticklabels(classes)
    ax.axhline(0, color="black", lw=0.5)
    ax.set_ylabel("AP delta (late - early)")
    ax.set_title("Per-class AP change: first vs last stream quartile")
    ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()
else:
    print("No checkpoint data with per-class AP found.")

## 10 Accept-rate dynamics

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))

for v in variants_main:
    if v not in RUN:
        continue
    fs = ah.read_csv(RUN[v] / "filter_stats.csv")
    if fs is None or fs.empty or "accept_rate" not in fs.columns:
        continue
    ax.plot(fs["items_processed"], fs["accept_rate"],
            label=v.replace("_", " "), color=PALETTE.get(v), lw=1.2)

ax.set_xlabel("items processed")
ax.set_ylabel("accept rate (per checkpoint interval)")
ax.set_title("Realized accept rate along the stream")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 11 Accept rate between refreshes

In [ ]:
refresh_curves: dict[str, pd.DataFrame] = {}
for v, rdir in RUN.items():
    cfg = ah.load_run_config(rdir)
    fam = ah.filter_mode(cfg)
    if fam not in {"window", "reservoir"}:
        continue
    dec = ENRICHED.get(v)
    if dec is None or dec.empty:
        continue
    seg = ah.refresh_accept_rate_segments(dec, ah.load_refreshes(rdir))
    if seg.empty:
        continue
    refresh_curves[v] = seg

if not refresh_curves:
    print("No window/reservoir runs with refreshes loaded.")
else:
    fig, ax = plt.subplots(figsize=(11, 4))
    for v, seg in refresh_curves.items():
        centers = (seg["segment_start"] + seg["segment_end"]) / 2
        ax.plot(centers, seg["accept_rate"], marker="o", ms=3, lw=1.2,
                color=PALETTE.get(v, "#333"), label=v)
    man = next(iter(MANIFESTS.values()), None)
    if man is not None:
        cfg0 = ah.load_run_config(next(iter(RUN.values())))
        for stream_idx, label in ah.block_transitions(
            man, bootstrap_frames=ah.get_bootstrap_size(man, cfg0),
        ):
            ax.axvline(stream_idx, color="grey", lw=0.4, alpha=0.4, ls="--")
    ax.set_xlabel("stream index (center of inter-refresh segment)")
    ax.set_ylabel("accept rate")
    ax.set_title("Accept rate per inter-refresh segment")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=7)
    plt.tight_layout()
    plt.show()

    print("\nSegment-level accept rates:")
    for v, seg in refresh_curves.items():
        print(f"\n-- {v} --")
        disp = seg.copy()
        disp["accept_rate"] = disp["accept_rate"].round(4)
        print(disp.to_string(index=False))

## 12 Per-block accept rate

In [ ]:
block_tables: dict[str, pd.DataFrame] = {}
for v, rdir in RUN.items():
    if v not in ENRICHED:
        continue
    cfg = ah.load_run_config(rdir)
    man = MANIFESTS.get(v)
    if not man:
        continue
    boot_n = ah.get_bootstrap_size(man, cfg)
    tbl = ah.per_block_accept_rate(ENRICHED[v], man, bootstrap_frames=boot_n)
    if tbl.empty:
        continue
    block_tables[v] = tbl
    print(f"\n=== {v}  (filter={ah.filter_mode(cfg)}, boot_n={boot_n}) ===")
    disp = tbl.copy()
    disp["accept_rate"] = disp["accept_rate"].round(4)
    print(disp.to_string(index=False))

In [ ]:
if block_tables:
    filter_runs = [v for v in block_tables if ah.filter_mode(
        ah.load_run_config(RUN[v])) in {"static", "window", "reservoir"}]
    if filter_runs:
        fig, ax = plt.subplots(figsize=(11, 4))
        width = 0.8 / max(1, len(filter_runs))
        ref_blocks = block_tables[filter_runs[0]]["block_label"].tolist()
        x = np.arange(len(ref_blocks))
        for i, v in enumerate(filter_runs):
            tbl = block_tables[v]
            if tbl["block_label"].tolist() != ref_blocks:
                continue
            ax.bar(x + i * width, tbl["accept_rate"].to_numpy(),
                   width=width, label=v, color=PALETTE.get(v, "#666"),
                   edgecolor="white", linewidth=0.4)
        ax.set_xticks(x + width * (len(filter_runs) - 1) / 2)
        ax.set_xticklabels(ref_blocks, rotation=45, ha="right")
        ax.set_ylabel("accept rate")
        ax.set_title("Per-block accept rate by filter variant")
        ax.grid(True, axis="y", alpha=0.3)
        ax.legend(fontsize=7, loc="upper right")
        plt.tight_layout()
        plt.show()
    else:
        print("No filter runs to plot (only no_filter / random).")

## 13 Per-domain acceptance

In [ ]:
# Per-domain accept rate bar chart
_domain_keys = list(variants_main)

if _domain_keys and any("scene_bucket" in ENRICHED.get(k, pd.DataFrame()).columns for k in _domain_keys):
    domain_rows: list[dict] = []
    for key in _domain_keys:
        df = ENRICHED.get(key, pd.DataFrame())
        if df.empty or "scene_bucket" not in df.columns or "action" not in df.columns:
            continue
        for bucket in df["scene_bucket"].dropna().unique():
            sub = df[df["scene_bucket"] == bucket]
            n_accept = int((sub["action"] == "accept").sum())
            n_total = len(sub)
            domain_rows.append({
                "variant": key, "domain": bucket,
                "accept_rate": n_accept / max(n_total, 1),
                "accepts": n_accept, "total": n_total,
            })
    domain_df = pd.DataFrame(domain_rows)

    if not domain_df.empty:
        domains = list(domain_df["domain"].unique())
        variants = list(domain_df["variant"].unique())
        fig, ax = plt.subplots(figsize=(max(6, 1.5 * len(domains) * len(variants)), 4))
        x = np.arange(len(domains))
        w = 0.8 / max(len(variants), 1)
        for i, v in enumerate(variants):
            sub = domain_df[domain_df["variant"] == v]
            rates = []
            for d in domains:
                sel = sub.loc[sub["domain"] == d, "accept_rate"]
                rates.append(float(sel.iloc[0]) if not sel.empty else 0.0)
            ax.bar(x + i * w, rates, w, label=v.replace("_", " "),
                   color=PALETTE.get(v, f"C{i}"), alpha=0.8)
        ax.axhline(0.5, color="grey", ls=":", lw=0.8, label="target (0.5)")
        ax.set_xticks(x + w * (len(variants) - 1) / 2)
        ax.set_xticklabels(domains, fontsize=8)
        ax.set_ylabel("accept rate")
        ax.set_title("Per-domain accept rate")
        ax.legend(fontsize=7, loc="best")
        plt.tight_layout()
        plt.show()

        print(domain_df.to_string(index=False))
else:
    print("No scene_bucket data in enriched decisions.")

## 14 Cumulative accepts

In [ ]:
# Cumulative accepts over stream index
fig, ax = plt.subplots(figsize=(10, 4))
for key in variants_main:
    df = ENRICHED.get(key, pd.DataFrame())
    if df.empty or "global_idx" not in df.columns or "action" not in df.columns:
        continue
    df_sorted = df.sort_values("global_idx")
    cum_accept = (df_sorted["action"] == "accept").cumsum()
    ax.plot(df_sorted["global_idx"].to_numpy(), cum_accept.to_numpy(),
            label=key.replace("_", " "), color=PALETTE.get(key), lw=1.2)

ax.set_xlabel("global_idx (stream order)")
ax.set_ylabel("cumulative accepts")
ax.set_title("Cumulative accepted frames along the stream")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 15 Acceptance-rate breakdown by metadata

In [ ]:
# Produce one set of plots per variant so each gets its own domain bands.
_dist_keys = [k for k in variants_main if k in ENRICHED]
_cp_interval = 1000

if not _dist_keys:
    print("No enriched decisions available yet.")

for _ekey in _dist_keys:
    _edf = ENRICHED[_ekey].sort_values("global_idx").copy()
    _edf["is_accept"] = (_edf["action"] == "accept").astype(int)
    _edf["window"] = _edf["global_idx"] // _cp_interval

    _window_stats: list[dict] = []
    for win_id, grp in _edf.groupby("window"):
        n = len(grp)
        accepts = grp["is_accept"].sum()
        _wid = int(win_id)  # type: ignore[arg-type]
        row: dict = {"window": _wid, "items_start": _wid * _cp_interval,
               "n": n, "accepts": accepts, "accept_rate": accepts / max(n, 1)}
        if "time_of_day" in grp.columns:
            for tod in ["day", "night", "twilight"]:
                row[f"frac_{tod}"] = (grp["time_of_day"] == tod).sum() / max(n, 1)
        if "road_condition" in grp.columns:
            for rc in ["wet", "snow"]:
                row[f"frac_{rc}"] = (grp["road_condition"] == rc).sum() / max(n, 1)
        if "scraped_weather" in grp.columns:
            row["frac_rain_snow"] = grp["scraped_weather"].isin(
                ["rain", "snow", "fog"]).sum() / max(n, 1)
        _window_stats.append(row)

    _ws = pd.DataFrame(_window_stats)
    _nice = _ekey.replace("_", " ")

    # --- 3-panel figure: accept rate + time-of-day + adverse conditions ---
    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

    ax0 = axes[0]
    ax0.plot(_ws["items_start"], _ws["accept_rate"], color="#d62728", lw=1.2, label="accept rate")
    ax0.set_ylabel("accept rate")
    ax0.set_title(f"Acceptance-rate breakdown -- {_nice}")
    ax0.set_ylim(-0.02, 1.02)
    ax0.legend(fontsize=7, loc="upper left")
    ax0.grid(True, alpha=0.3)
    add_block_bands(ax0, variant=_ekey)

    ax1 = axes[1]
    if "frac_night" in _ws.columns:
        ax1.fill_between(_ws["items_start"], 0, _ws["frac_night"],
                         alpha=0.6, color="#2c3e50", label="night")
        ax1.fill_between(_ws["items_start"], _ws["frac_night"],
                         _ws["frac_night"] + _ws.get("frac_twilight", 0),
                         alpha=0.5, color="#e67e22", label="twilight")
        ax1.fill_between(_ws["items_start"],
                         _ws["frac_night"] + _ws.get("frac_twilight", 0), 1.0,
                         alpha=0.3, color="#f1c40f", label="day")
    ax1.set_ylabel("fraction of frames")
    ax1.set_title(f"Time-of-day composition -- {_nice}")
    ax1.legend(fontsize=7, loc="upper left")
    ax1.set_ylim(0, 1)
    ax1.grid(True, alpha=0.2)
    add_block_bands(ax1, variant=_ekey)

    ax2 = axes[2]
    if "frac_wet" in _ws.columns:
        ax2.plot(_ws["items_start"], _ws["frac_wet"], label="wet road", lw=1.1, color="#3498db")
    if "frac_snow" in _ws.columns:
        ax2.plot(_ws["items_start"], _ws["frac_snow"], label="snow road", lw=1.1, color="#95a5a6")
    if "frac_rain_snow" in _ws.columns:
        ax2.plot(_ws["items_start"], _ws["frac_rain_snow"],
                 label="rain/snow/fog weather", lw=1.1, color="#8e44ad", ls="--")
    ax2.set_ylabel("fraction of frames")
    ax2.set_xlabel("items processed")
    ax2.set_title(f"Adverse conditions -- {_nice}")
    ax2.legend(fontsize=7, loc="upper left")
    ax2.set_ylim(-0.02, 1.02)
    ax2.grid(True, alpha=0.3)
    add_block_bands(ax2, variant=_ekey)

    plt.tight_layout()
    plt.show()

    # --- Conditional acceptance rate by time_of_day ---
    if "time_of_day" in _edf.columns:
        fig2, ax_cond = plt.subplots(figsize=(14, 3.5))
        for tod, color in [("day", "#f1c40f"), ("night", "#2c3e50"), ("twilight", "#e67e22")]:
            sub = _edf[_edf["time_of_day"] == tod].copy()
            if len(sub) < 50:
                continue
            win = min(500, max(20, len(sub) // 20))
            accept_s = pd.Series(sub["is_accept"])
            roll = accept_s.rolling(win, min_periods=10).mean()
            ax_cond.plot(np.asarray(sub["global_idx"]), np.asarray(roll),
                         label=f"{tod} (n={len(sub):,})", color=color, lw=1.1)
        ax_cond.set_xlabel("global_idx (stream order)")
        ax_cond.set_ylabel("rolling accept rate")
        ax_cond.set_title(f"Acceptance rate by time_of_day -- {_nice}")
        ax_cond.set_ylim(-0.02, 1.02)
        ax_cond.legend(fontsize=7)
        ax_cond.grid(True, alpha=0.3)
        add_block_bands(ax_cond, variant=_ekey)
        plt.tight_layout()
        plt.show()

## 16 Score separation

In [ ]:
def plot_score_violins(label: str, df: pd.DataFrame, ax: Axes) -> None:
    if "filter_score" not in df.columns or "action" not in df.columns:
        ax.set_title(f"{label}: missing columns")
        return
    sub = ah.subsample_decisions(df, min(SUBSAMPLE_N, len(df)), RNG)
    parts = []
    for act, color in [("accept", "#2ca02c"), ("reject", "#d62728")]:
        s = sub.loc[sub["action"] == act, "filter_score"].astype(float)
        if len(s) > 0:
            parts.append((act, s.to_numpy(dtype=float, copy=False), color))
    if not parts:
        return
    positions = list(range(1, len(parts) + 1))
    vp = ax.violinplot([p[1] for p in parts], positions=positions,
                       widths=0.7, showmeans=True, showextrema=True)
    for i, body in enumerate(vp["bodies"]):  # type: ignore[arg-type]
        body.set_facecolor(parts[i][2])
        body.set_alpha(0.55)
    ax.set_xticks(positions)
    ax.set_xticklabels([p[0] for p in parts])
    ax.set_ylabel("filter_score")
    ax.set_title(label.replace("_", " "))


# Any run that logs a meaningful per-frame score (Mahalanobis distance
# or detection uncertainty).  Pure no-filter / random runs are skipped
# because their filter_score is trivially zero.
def _has_score(key: str) -> bool:
    df = ENRICHED[key]
    if "filter_score" not in df.columns:
        return False
    cfg = ah.load_run_config(RUN[key])
    return ah.filter_mode(cfg) in {"static", "window", "reservoir", "uncertainty"}


score_keys = [k for k in ENRICHED if _has_score(k)]
if not score_keys:
    print("No score-based runs loaded.")
else:
    n_sk = len(score_keys)
    n_cols = min(n_sk, 4)
    fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 3.2), squeeze=False)
    for a, key in zip(axes[0], score_keys[:4]):
        plot_score_violins(key, ENRICHED[key], a)
    fig.suptitle("Filter score by decision (subsampled)", y=1.02)
    plt.tight_layout()
    plt.show()

## 17 Category-level acceptance

In [ ]:
TARGET_CLASSES = ["Vehicle", "Pedestrian", "VulnerableVehicle"]
# Every filter run (static / window / reservoir / uncertainty); skip
# no_filter and random because their decisions are domain-independent.
cat_keys = [k for k in ENRICHED if ah.filter_mode(
    ah.load_run_config(RUN[k])) in {"static", "window", "reservoir", "uncertainty"}]

if not cat_keys:
    print("No filter runs with decisions loaded.")
else:
    rows = []
    for key in cat_keys:
        df = ENRICHED[key]
        if "categories" not in df.columns:
            continue
        for cls in TARGET_CLASSES:
            has_cls = df["categories"].fillna("").str.contains(cls)
            for action in ("accept", "reject"):
                mask = df["action"] == action
                n_with = int((mask & has_cls).sum())
                n_total = int(mask.sum())
                rows.append({"variant": key, "class": cls, "action": action,
                             "count": n_with, "fraction": n_with / max(n_total, 1)})
    cat_df = pd.DataFrame(rows)

    fig, axes = plt.subplots(1, len(TARGET_CLASSES), figsize=(4 * len(TARGET_CLASSES), 3.5), squeeze=False)
    for i, cls in enumerate(TARGET_CLASSES):
        ax = axes[0][i]
        sub = cat_df[cat_df["class"] == cls]
        x = np.arange(len(cat_keys))
        w = 0.35
        for j, action in enumerate(("accept", "reject")):
            vals = []
            for k in cat_keys:
                sel = sub.loc[(sub["variant"] == k) & (sub["action"] == action), "fraction"]
                vals.append(float(sel.iloc[0]) if len(sel) > 0 else 0.0)
            color = "#2ca02c" if action == "accept" else "#d62728"
            ax.bar(x + j * w, vals, w, label=action, color=color, alpha=0.7)
        ax.set_xticks(x + w / 2)
        ax.set_xticklabels([k.replace("_", "\n") for k in cat_keys], fontsize=7)
        ax.set_ylabel("fraction of decisions containing class")
        ax.set_title(cls)
        if i == 0:
            ax.legend(fontsize=7)
    fig.suptitle("Category presence in accepted vs rejected frames", y=1.04)
    plt.tight_layout()
    plt.show()

## 18 Inter-accept gaps

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.5))

for key in variants_main:
    if key not in ENRICHED:
        continue
    gaps = ah.inter_accept_gaps(ENRICHED[key])
    if gaps.empty:
        continue
    arr = gaps.to_numpy(dtype=float)
    cap = float(np.nanpercentile(arr, 99))
    g = np.minimum(arr, cap)
    ax.hist(g, bins=60, density=True, alpha=0.45,
            label=key.replace("_", " "), color=PALETTE.get(key))

ax.set_xlabel("gap in global_idx until next accept")
ax.set_ylabel("density")
ax.legend(fontsize=7)
ax.set_title("Inter-accept spacing (clipped at 99th percentile)")
plt.tight_layout()
plt.show()

## 19 Object-count scatter

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

for key in variants_main:
    if key not in ENRICHED:
        continue
    sub = ENRICHED[key].loc[ENRICHED[key]["action"] == "accept"]
    sub = ah.subsample_decisions(sub, min(8000, len(sub)), RNG)
    if "global_idx" not in sub.columns or "num_objects" not in sub.columns:
        continue
    ax.scatter(sub["global_idx"], sub["num_objects"], s=6, alpha=0.2, rasterized=True,
               c=PALETTE.get(key), label=key.replace("_", " "))

ax.set_xlabel("global_idx (stream order)")
ax.set_ylabel("num_objects (from manifest)")
ax.legend(markerscale=3, fontsize=7)
ax.set_title("Accepted frames: object density vs stream position")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 20 Scoring-model refresh timeline

In [ ]:
adaptive_keys = [k for k in RUN if k.startswith("adaptive_")]

if not adaptive_keys:
    print("No adaptive streaming runs loaded.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))

    for v in adaptive_keys:
        ref = ah.read_csv(RUN[v] / "refreshes.csv")
        if ref is None or ref.empty:
            continue
        color = PALETTE.get(v, "#333")
        label = v.replace("adaptive_", "").replace("_", " ")

        # Threshold trajectory: step from threshold_before (at items_seen)
        # to threshold_after, then hold until the next refresh.
        xs = [0.0]
        ys = [float(ref["threshold_before"].iloc[0])]
        for _, r in ref.iterrows():
            xs += [float(r["items_seen"]), float(r["items_seen"])]
            ys += [float(r["threshold_before"]), float(r["threshold_after"])]
        axes[0].plot(xs, ys, color=color, lw=1.3, label=label)

        axes[1].plot(ref["refresh_idx"].to_numpy(),
                     ref["duration_seconds"].to_numpy(),
                     marker="o", ms=4, color=color, label=label)

    axes[0].set(xlabel="items processed", ylabel="filter threshold",
                title="Adaptive threshold evolution (Mahalanobis or uncertainty)")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(fontsize=7)

    axes[1].set(xlabel="refresh index", ylabel="duration (s)",
                title="Refresh latency")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()